# Exercise 1: Language family exploration

*The Turkic World · version 0.1.0*

> Choose a Turkic language you are unfamiliar with from the language table. Using online resources, find its Wikipedia edition (if it exists) and estimate its digital footprint. How many Wikipedia articles does it have? Is it covered by OSCAR or FineWeb2? Does it have a Universal Dependencies treebank? Locate the language on the map and identify its branch according to the branch table.

The worked solution uses **Turkmen**. Change `LANG` to explore another language.

**Setup.** `pip install "turkicnlp==0.5.0" requests pandas`. Cells that query websites need internet access; if a request fails, the cell records the failure instead of a number. An unavailable value is not zero.

In [ ]:
import datetime
import requests
import pandas as pd

ACCESSED = datetime.date.today().isoformat()        # record the access date with every observation
HEADERS = {"User-Agent": "turkic-nlp-book-exercises/0.1 (educational use)"}

def get_json(url, **params):
    """Return parsed JSON, or {'_error': ...} if the request fails (never a fake value)."""
    try:
        r = requests.get(url, params=params or None, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as exc:
        return {"_error": f"{type(exc).__name__}: {exc}"}

def get_text(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return r.text
    except Exception as exc:
        return None

def github_org_repos(org):
    """All public repositories of a GitHub organisation (paged listing; avoids the search API's low rate limit)."""
    repos, page = [], 1
    while True:
        d = get_json(f"https://api.github.com/orgs/{org}/repos", per_page=100, page=page)
        if not isinstance(d, list):
            return repos, d.get("_error")
        repos += d
        if len(d) < 100:
            return repos, None
        page += 1

In [ ]:
# Branch assignments from The Turkic World's branch table (Johanson 2021, pp. 22–23)
BRANCH = {
    "Turkish": "SW (Oghuz)", "Azerbaijani": "SW (Oghuz)", "Gagauz": "SW (Oghuz)", "Turkmen": "SW (Oghuz)",
    "Kazakh": "NW (Kipchak)", "Kyrgyz": "NW (Kipchak)", "Tatar": "NW (Kipchak)", "Bashkir": "NW (Kipchak)",
    "Karakalpak": "NW (Kipchak)", "Kumyk": "NW (Kipchak)", "Nogai": "NW (Kipchak)",
    "Crimean Tatar": "NW (Kipchak)", "Karachay-Balkar": "NW (Kipchak)",
    "Uzbek": "SE (Karluk)", "Uyghur": "SE (Karluk)",
    "Sakha (Yakut)": "NE (Siberian)", "Tuvan": "NE (Siberian)", "Khakas": "NE (Siberian)",
    "Shor": "NE (Siberian)", "Altai": "NE (Siberian)",
    "Chuvash": "Oghur",
}

# Identifiers differ between resources: record them explicitly
LANG = {"name": "Turkmen", "wikipedia": "tk", "fineweb2": "tuk_Latn", "ud_name": "Turkmen"}
print(LANG["name"], "→ branch:", BRANCH[LANG["name"]])

In [ ]:
from pathlib import Path
from IPython.display import Image, display

map_path = Path("../images/Turkic_Languages_distribution_map.png")   # the map used in The Turkic World
if map_path.exists():
    display(Image(filename=str(map_path), width=700))
else:
    print("Map not found; see https://commons.wikimedia.org/wiki/File:Turkic_Languages_distribution_map.png")

In [ ]:
rows = []

# 1. Wikipedia edition size (MediaWiki API)
wiki_url = f"https://{LANG['wikipedia']}.wikipedia.org/w/api.php"
d = get_json(wiki_url, action="query", meta="siteinfo", siprop="statistics", format="json")
articles = d.get("query", {}).get("statistics", {}).get("articles")
rows.append({"item": "Wikipedia articles", "finding": articles if articles is not None else f"not retrieved ({d.get('_error')})",
             "source": wiki_url, "accessed": ACCESSED,
             "limitation": "article count, not words; includes stubs and bot-created pages"})

# 2. FineWeb2 coverage (Hugging Face dataset viewer API)
fw_url = "https://datasets-server.huggingface.co/size"
d = get_json(fw_url, dataset="HuggingFaceFW/fineweb-2", config=LANG["fineweb2"])
if "size" in d:
    splits = {s["split"]: s["num_rows"] for s in d["size"]["splits"]}
    finding = f"config {LANG['fineweb2']}: " + ", ".join(f"{k} {v:,} documents" for k, v in splits.items())
else:
    finding = f"not retrieved ({d.get('_error')})"
rows.append({"item": "FineWeb2", "finding": finding, "source": f"{fw_url}?dataset=HuggingFaceFW/fineweb-2&config={LANG['fineweb2']}",
             "accessed": ACCESSED, "limitation": "document counts; quality and language-ID accuracy not checked"})

# 3. Universal Dependencies treebanks (GitHub organisation of the UD project)
all_ud, ud_error = github_org_repos("UniversalDependencies")
repos = [r for r in all_ud if r["name"].startswith(f"UD_{LANG['ud_name']}-")]
for r in repos:
    readme = get_text(f"https://raw.githubusercontent.com/UniversalDependencies/{r['name']}/{r['default_branch']}/README.md") or ""
    meta = {k.strip(): v.strip() for k, v in (l.split(":", 1) for l in readme.splitlines()
            if ":" in l and l.split(":", 1)[0] in ("Data available since", "License", "Lemmas", "Relations"))}
    rows.append({"item": f"UD treebank {r['name']}", "finding": "; ".join(f"{k}: {v}" for k, v in meta.items()) or "README metadata not found",
                 "source": r["html_url"], "accessed": ACCESSED,
                 "limitation": "check the release you use; a repository is not necessarily in every release"})
if not repos:
    rows.append({"item": "UD treebank", "finding": f"none found by this search ({ud_error or 'no matching repository'})",
                 "source": "https://github.com/UniversalDependencies", "accessed": ACCESSED,
                 "limitation": "a failed search is not proof of absence"})

pd.set_option("display.max_colwidth", 120)
pd.DataFrame(rows)

## Worked solution (Turkmen, checked 18 October 2026)

- **Branch and location.** Turkmen belongs to the SW (Oghuz) branch with Turkish, Azerbaijani and Gagauz. On the map it is spoken east of the Caspian Sea, in Turkmenistan and neighbouring regions of Iran and Afghanistan.
- **Wikipedia.** The Turkmen edition is `tk.wikipedia.org`; the cell above reads its live article count. Article counts include stubs and bot-generated pages, so they are a rough indicator of digital presence, not of corpus size.
- **FineWeb2.** Covered as `tuk_Latn`: 238,155 training and 2,269 test documents (dataset viewer, 18 October 2026). These are documents, not words, and their quality is not established by the count.
- **OSCAR.** Not queried here: recent OSCAR releases require accepting access conditions on Hugging Face. Check the release notes manually and record the version.
- **Universal Dependencies.** `UD_Turkmen-TUD` exists; its README states *Data available since: UD v2.18* and licence CC BY-SA 4.0. The book's inventory is pinned to UD 2.17, where Turkmen has no treebank. This is why every resource statement needs a release and a date.

**Interpretation.** Turkmen has a mid-sized web footprint and, since UD 2.18, a treebank. The counts say little about quality, domain or licence, which the resource audit in The Turkic World asks you to check next.

## Your turn — extension

Set `LANG` to another language from the table (for example `{"name": "Bashkir", "wikipedia": "ba", "fineweb2": "bak_Cyrl", "ud_name": "Bashkir"}`), rerun the cells and write your interpretation here.